In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

In [ ]:
res = 0.1

In [ ]:
type_index = 1

In [ ]:
# points, edges = pattern_generator_using_gmsh.get_four_square(5, type_index, avg_len_boundary = res, avg_len_embeddings = res)



In [ ]:
# visualization.plot_line_segments(points, edges - 1)

In [ ]:
isheet, m, marker = pattern_generator_using_gmsh.get_four_square(5, type_index, avg_len_boundary = res, avg_len_embeddings = res)


In [ ]:
type_index

In [ ]:
visualization.plot_2d_mesh(m, pointList = marker, width = 5, height = 5)

In [ ]:
viewer = TriMeshViewer(isheet, width=768, height=640)


In [ ]:
viewer.showWireframe(False)

In [ ]:
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.1


In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
from matplotlib import cm

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(isheet)[:, 0])

In [ ]:
# Choose strategy for constraining rigid motion
if not allowBending:
    fixedVars, hessianShift = [isheet.numVars() - 2, isheet.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

isheet.setUseTensionFieldEnergy(useTFT)
isheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    isheet.disableFusedRegionTensionFieldTheory(False)
isheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])
benchmark.report()

In [ ]:
max(utils.getStrains(isheet)[:, 0])

In [ ]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(ipu.sheet)[:, 0], bins=1000);
plt.xlim(0, max(utils.getStrains(ipu.sheet)[:, 0]));

In [ ]:
strains = utils.getStrains(ipu.sheet)[:, 0]
strainField = vis.fields.ScalarField(ipu.sheet, strains, colormap = cm.viridis, vmin= 0, vmax = max(utils.getStrains(ipu.sheet)[:, 0]))

In [ ]:
viewer.update(scalarField=strainField)

In [ ]:
viewer.saveColorizedObj("four_square_pillow_{}.obj".format(type_index))

In [ ]:
ipu.energy(inflation.InflatableSheet.EnergyType.Elastic), ipu.energy(inflation.InflatableSheet.EnergyType.Pressure)